# FEDformer V2 for UCR Classification

This notebook keeps the FEDformer-specific scientific component while aligning the experiment protocol with `train_all_twotower_v2`: deterministic train/validation split, one-hot labels with BCEWithLogitsLoss, validation-loss checkpoint selection, corrected binary/multiclass metrics, one official TEST evaluation, resumable outputs, and failure logs.

The original FEDformer paper targets long-term forecasting and uses decomposition plus frequency-enhanced blocks in an encoder-decoder forecasting architecture. UCR is a classification task, so this implementation uses the FEDformer encoder-side ideas (moving-average series decomposition and Fourier-enhanced frequency modes) followed by a classification pooling head. It is a faithful task adaptation, not a claim that the original forecasting decoder is being used unchanged.

In [ ]:
# Cell 1 - imports and configuration
from __future__ import annotations
import copy, json, os, platform, random, sys, traceback
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset

DATA_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\UCRArchive_2018")
OUTPUT_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2")
MODEL_NAME = "fedformer"
EXPERIMENT_VERSION = "fedformer_v2_bce_correct_metrics"
SEED = 42
BATCH_SIZE = 8
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
T_MAX = 50
ETA_MIN = 0.0
VAL_FRACTION = 0.20
MIN_EFFECTIVE_LENGTH = 8
NUM_WORKERS = 0
D_MODEL = 256           # alternatives: 128, 512
E_LAYERS = 2            # alternatives: 1, 3, 4
D_FF = 512              # alternatives: 256, 1024
DROPOUT = 0.1           # alternatives: 0.0, 0.2, 0.3
FOURIER_MODES = 8       # alternatives: 16, 32
DECOMP_KERNEL = 25      # alternatives: 13, 49; use a positive odd integer
DEVICE_REQUEST = "cuda:0"
REQUIRE_AOUT_MARKERS = True  # keeps the same requested dataset universe as V2
RUN_SMOKE_TEST = False
RUN_FULL_EXPERIMENT = True
SMOKE_DATASETS = ["Coffee", "ArrowHead"]
SMOKE_EPOCHS = 3
RESUME_COMPLETED_DATASETS = True
SAVE_INDIVIDUAL_CURVES = True

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True; cudnn.benchmark = False
set_global_seed(SEED)
DEVICE = torch.device(DEVICE_REQUEST if DEVICE_REQUEST.startswith("cuda") and torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Device:", DEVICE)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 - V2 persistence and metadata
def model_output_dir(debug=False):
    if debug:
        return OUTPUT_ROOT / "_debug" / MODEL_NAME / f"seed_{SEED}"
    return OUTPUT_ROOT / MODEL_NAME / f"seed_{SEED}"

def ensure_output_tree(base_dir):
    paths = {"base": base_dir, "checkpoints": base_dir/"checkpoints", "curves": base_dir/"curves", "dataset_histories": base_dir/"dataset_histories", "shared_splits": OUTPUT_ROOT/"_shared_splits"/f"seed_{SEED}"}
    for p in paths.values(): p.mkdir(parents=True, exist_ok=True)
    return paths

def atomic_write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_suffix(path.suffix+".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8"); os.replace(tmp, path)

def atomic_write_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_suffix(path.suffix+".tmp")
    with open(tmp, "w", encoding="utf-8") as f: json.dump(payload, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def append_or_replace(path, new_df, dataset_name):
    old = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if not old.empty and "dataset" in old.columns: old = old[old.dataset != dataset_name]
    out = pd.concat([old, new_df], ignore_index=True)
    keys = [c for c in ["dataset", "epoch"] if c in out.columns]
    if keys: out = out.sort_values(keys).reset_index(drop=True)
    atomic_write_csv(out, path); return out

def run_config(debug, epochs):
    return {"experiment_version": EXPERIMENT_VERSION, "model": MODEL_NAME, "seed": SEED, "debug": bool(debug), "data_root": str(DATA_ROOT), "output_root": str(OUTPUT_ROOT), "batch_size": BATCH_SIZE, "num_epochs": int(epochs), "learning_rate": LEARNING_RATE, "scheduler": "CosineAnnealingLR", "loss": "BCEWithLogitsLoss", "label_encoding": "one_hot_or_binary_column", "checkpoint_selection": "minimum validation BCE loss", "test_evaluation": "once after restoring best checkpoint", "decomposition": "moving-average seasonal-trend decomposition", "frequency_block": "Fourier-enhanced fixed low-frequency modes", "python": platform.python_version(), "pytorch": torch.__version__, "created_utc": datetime.now(timezone.utc).isoformat()}

def make_class_safe_split(labels, dataset_name, split_dir, val_fraction=0.2):
    split_dir.mkdir(parents=True, exist_ok=True); path = split_dir/f"{dataset_name}_split.npz"; labels=np.asarray(labels)
    if path.exists():
        z=np.load(path); return z["train_idx"].astype(int), z["val_idx"].astype(int), path
    rng=np.random.default_rng(SEED); tr=[]; va=[]
    for value in np.unique(labels):
        idx=rng.permutation(np.flatnonzero(labels==value)); n=0 if len(idx)<=1 else min(len(idx)-1,max(1,int(round(len(idx)*val_fraction))))
        va.append(idx[:n]); tr.append(idx[n:])
    train_idx=np.sort(np.concatenate(tr)).astype(int); val_idx=np.sort(np.concatenate([x for x in va if len(x)] )).astype(int)
    np.savez_compressed(path, train_idx=train_idx, val_idx=val_idx, n_samples=len(labels), seed=SEED); return train_idx,val_idx,path

In [ ]:
# Cell 3 - preprocessing and FEDformer classification architecture
def clean_and_pad(raw, min_len=8, fixed_len=None):
    rows=[]; keep=[]; lengths=[]
    for i,row in enumerate(raw):
        v=row[~np.isnan(row)]; L=len(v)
        if L<min_len: continue
        mu=float(v.mean()); sd=float(v.std()); v=(v-mu)/(sd if sd>0 else 1.0)
        rows.append(v); keep.append(i); lengths.append(L)
    if not rows: raise ValueError("All samples were filtered out.")
    target=int(fixed_len if fixed_len is not None else max(lengths)); out=[]
    for v in rows: out.append(v[:target] if len(v)>=target else np.pad(v,(0,target-len(v))))
    return np.stack(out).astype("float32"), np.asarray(keep,dtype=int)

class SeriesDataset(Dataset):
    def __init__(self,x,y): self.x=x.astype("float32"); self.y=y.astype("float32")
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]

class MovingAverage(nn.Module):
    def __init__(self,kernel): super().__init__(); self.avg=nn.AvgPool1d(kernel, stride=1, padding=0)
    def forward(self,x):
        pad=(self.avg.kernel_size[0]-1)//2; front=x[:,0:1,:].repeat(1,pad,1); end=x[:,-1:,:].repeat(1,pad,1)
        return self.avg(torch.cat([front,x,end],dim=1).transpose(1,2)).transpose(1,2)

class SeriesDecomp(nn.Module):
    def __init__(self,kernel): super().__init__(); self.moving=MovingAverage(kernel)
    def forward(self,x): trend=self.moving(x); return x-trend, trend

class FourierEnhancedBlock(nn.Module):
    """Learnable Fourier modes, following the frequency-enhanced idea of FEDformer."""
    def __init__(self,seq_len,d_model,modes=16):
        super().__init__(); nfreq=seq_len//2+1; m=min(int(modes),nfreq); self.register_buffer("indices",torch.arange(m)); self.weight=nn.Parameter(torch.randn(m,d_model,2)*0.02)
    def forward(self,x):
        xf=torch.fft.rfft(x,dim=1); out=torch.zeros_like(xf); w=torch.view_as_complex(self.weight.contiguous()); idx=self.indices.to(xf.device); out[:,idx,:]=xf[:,idx,:]*w.to(xf.dtype).unsqueeze(0); return torch.fft.irfft(out,n=x.size(1),dim=1)

class FEDEncoderLayer(nn.Module):
    def __init__(self,seq_len,d_model=256,d_ff=512,dropout=0.1,modes=16,kernel=25):
        super().__init__(); self.decomp=SeriesDecomp(kernel); self.freq=FourierEnhancedBlock(seq_len,d_model,modes); self.norm1=nn.LayerNorm(d_model); self.norm2=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout); self.ff=nn.Sequential(nn.Linear(d_model,d_ff),nn.GELU(),nn.Dropout(dropout),nn.Linear(d_ff,d_model))
    def forward(self,x):
        seasonal,trend=self.decomp(x); seasonal=self.norm1(seasonal+self.drop(self.freq(seasonal))); seasonal=self.norm2(seasonal+self.drop(self.ff(seasonal))); return seasonal+trend

class FEDformerEncoderClassifier(nn.Module):
    def __init__(self,seq_len,num_outputs,d_model=256,e_layers=2,d_ff=512,dropout=0.1,modes=16,kernel=25):
        super().__init__(); self.token=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.zeros(1,seq_len,d_model)); nn.init.trunc_normal_(self.pos,std=0.02); k=min(kernel,seq_len if seq_len%2 else seq_len-1); k=max(3,k); self.layers=nn.ModuleList([FEDEncoderLayer(seq_len,d_model,d_ff,dropout,modes,k) for _ in range(e_layers)]); self.drop=nn.Dropout(dropout); self.fc=nn.Linear(d_model,num_outputs)
    def forward(self,x):
        if x.ndim==2: x=x.unsqueeze(-1)
        h=self.token(x)+self.pos[:,:x.size(1),:]
        for layer in self.layers: h=layer(h)
        return self.fc(self.drop(h.mean(dim=1)))

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Cell 4 - corrected metrics and evaluation helpers
def macro_auc(labels, probs):
    values=[]
    for j in range(labels.shape[1]):
        if np.unique(labels[:,j]).size==2: values.append(roc_auc_score(labels[:,j],probs[:,j]))
    return float(np.mean(values)) if values else float("nan")

def corrected_metrics(logits,labels):
    if logits.ndim==1: logits=logits[:,None]
    if labels.ndim==1: labels=labels[:,None]
    probs=1/(1+np.exp(-np.clip(logits,-50,50)))
    if logits.shape[1]==1:
        yt=labels[:,0].astype(int); yp=(probs[:,0]>=0.5).astype(int); acc=accuracy_score(yt,yp); auc=roc_auc_score(yt,probs[:,0]) if np.unique(yt).size==2 else float("nan")
    else:
        yt=np.argmax(labels,axis=1); yp=np.argmax(logits,axis=1); acc=accuracy_score(yt,yp); auc=macro_auc(labels,probs)
    return float(auc),float(acc)

def evaluate_loss(model,loader,criterion):
    model.eval(); total=0.0; n=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); y=y.to(DEVICE); loss=criterion(model(x),y); total+=float(loss.item())*len(y); n+=len(y)
    return total/n

def evaluate_test_once(model,loader,criterion):
    model.eval(); logits=[]; labels=[]; total=0.0; n=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); yd=y.to(DEVICE); out=model(x); total+=float(criterion(out,yd).item())*len(y); n+=len(y); logits.append(out.cpu().numpy()); labels.append(y.numpy())
    auc,acc=corrected_metrics(np.concatenate(logits),np.concatenate(labels)); return auc,acc,total/n,n

# Self-checks: binary sigmoid threshold and multiclass argmax
assert np.isclose(corrected_metrics(np.array([[-2.],[2.],[1.],[-1.]]),np.array([[0.],[1.],[0.],[1.]]))[1],0.5)
assert tuple(FEDformerEncoderClassifier(16,1,d_model=16,d_ff=32,modes=4,kernel=5)(torch.randn(2,16,1)).shape)==(2,1)
print("FEDformer V2 self-checks passed.")

In [ ]:
# Cell 5 - one complete dataset with the unified V2 protocol
def run_one_dataset_v2(dataset_dir,dataset_name,paths,num_epochs=NUM_EPOCHS,save_curve=True):
    set_global_seed(SEED); train_tsv=dataset_dir/f"{dataset_name}_TRAIN_cleaned.tsv"; test_tsv=dataset_dir/f"{dataset_name}_TEST_cleaned.tsv"
    required=[train_tsv,test_tsv]
    if REQUIRE_AOUT_MARKERS: required += [dataset_dir/f"{dataset_name}_Aout_train_k2.csv",dataset_dir/f"{dataset_name}_Aout_test_k2.csv"]
    missing=[str(p) for p in required if not p.exists()]
    if missing: raise FileNotFoundError('Missing required files: ' + '; '.join(missing))
    tr=pd.read_csv(train_tsv,sep="\t",header=None); te=pd.read_csv(test_tsv,sep="\t",header=None)
    ytr_raw=tr.iloc[:,0].to_numpy(); yte_raw=te.iloc[:,0].to_numpy(); xtr_raw=tr.iloc[:,1:].to_numpy(dtype="float32"); xte_raw=te.iloc[:,1:].to_numpy(dtype="float32")
    temp,_=clean_and_pad(xtr_raw,MIN_EFFECTIVE_LENGTH,None); seq_len=temp.shape[1]
    xtr,keep_tr=clean_and_pad(xtr_raw,MIN_EFFECTIVE_LENGTH,seq_len); xte,keep_te=clean_and_pad(xte_raw,MIN_EFFECTIVE_LENGTH,seq_len)
    ytr=ytr_raw[keep_tr]; yte=yte_raw[keep_te]; classes=np.sort(np.unique(ytr))
    if len(np.setdiff1d(np.unique(yte),classes)): raise ValueError("TEST contains unseen labels.")
    Ytr=label_binarize(ytr,classes=classes).astype("float32"); Yte=label_binarize(yte,classes=classes).astype("float32")
    if Ytr.ndim==1: Ytr=Ytr[:,None]; Yte=Yte[:,None]
    train_idx,val_idx,split_path=make_class_safe_split(ytr,dataset_name,paths["shared_splits"],VAL_FRACTION)
    xtr=xtr[:,:,None]; xte=xte[:,:,None]; train_ds=SeriesDataset(xtr[train_idx],Ytr[train_idx]); val_ds=SeriesDataset(xtr[val_idx],Ytr[val_idx]); test_ds=SeriesDataset(xte,Yte)
    gen=torch.Generator().manual_seed(SEED); pin=DEVICE.type=="cuda"
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,generator=gen,num_workers=NUM_WORKERS,pin_memory=pin); val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=pin); test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=pin)
    model=FEDformerEncoderClassifier(seq_len,int(Ytr.shape[1]),D_MODEL,E_LAYERS,D_FF,DROPOUT,FOURIER_MODES,DECOMP_KERNEL).to(DEVICE); params=count_params(model)
    print(f"[{dataset_name}] model=FEDformerEncoderClassifier | uses_aout=False | decomposition=moving_average | frequency_block=FourierEnhanced | outputs={Ytr.shape[1]} | classes={len(classes)} | params={params:,} | train/val/test={len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    criterion=nn.BCEWithLogitsLoss(); opt=optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY); sched=CosineAnnealingLR(opt,T_max=T_MAX,eta_min=ETA_MIN); best=float("inf"); best_epoch=0; hist=[]; ckpt=paths["checkpoints"]/f"{dataset_name}_best.pt"
    for epoch in range(1,num_epochs+1):
        model.train(); total=0.0; n=0
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); opt.step(); total+=float(loss.item())*len(y); n+=len(y)
        train_loss=total/n; val_loss=evaluate_loss(model,val_loader,criterion); hist.append({"dataset":dataset_name,"model":MODEL_NAME,"seed":SEED,"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"learning_rate":opt.param_groups[0]["lr"]})
        if val_loss<best:
            best=val_loss; best_epoch=epoch; torch.save({"state_dict":copy.deepcopy(model.state_dict()),"best_epoch":epoch,"best_val_loss":best,"classes":classes.tolist(),"seq_len":int(seq_len),"parameter_count":params,"split_path":str(split_path)},ckpt)
        sched.step()
    history=pd.DataFrame(hist); atomic_write_csv(history,paths["dataset_histories"]/f"{dataset_name}.csv"); model.load_state_dict(torch.load(ckpt,map_location=DEVICE)["state_dict"]); auc,acc,loss,n_test=evaluate_test_once(model,test_loader,criterion)
    if save_curve:
        fig,ax=plt.subplots(figsize=(6.2,4)); ax.plot(history.epoch,history.train_loss,label="Train BCE"); ax.plot(history.epoch,history.val_loss,label="Validation BCE"); ax.axvline(best_epoch,color="black",ls="--",lw=1,label="Best epoch"); ax.set(title=f"{dataset_name} - FEDformer V2",xlabel="Epoch",ylabel="BCE loss"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(paths["curves"]/f"{dataset_name}_loss.png",dpi=180); plt.close(fig)
    result={"dataset":dataset_name,"model":MODEL_NAME,"seed":SEED,"test_auc":auc,"test_acc":acc,"test_loss":loss,"n_samples":n_test,"num_classes":len(classes),"num_outputs":Ytr.shape[1],"best_epoch":best_epoch,"best_val_loss":best,"train_samples":len(train_ds),"val_samples":len(val_ds),"parameter_count":params,"uses_aout":False,"decomposition":"moving_average","frequency_block":"FourierEnhanced","status":"completed"}
    print(f"[{dataset_name}] TEST loss={loss:.4f} | AUC={auc:.4f} | ACC={acc:.4f} | best_epoch={best_epoch} | n={n_test}")
    return result,history

In [ ]:
# Cell 6 - discovery, resumable runner, summaries, and failure recording
def discover_datasets(root):
    names=[]
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        n=d.name; req=[d/f"{n}_TRAIN_cleaned.tsv",d/f"{n}_TEST_cleaned.tsv"]
        if REQUIRE_AOUT_MARKERS: req += [d/f"{n}_Aout_train_k2.csv",d/f"{n}_Aout_test_k2.csv"]
        if all(p.exists() for p in req): names.append(n)
    return names

def summarize(df):
    return {"datasets_completed":int(len(df)),"simple_mean_auc":float(df.test_auc.mean()),"simple_mean_acc":float(df.test_acc.mean()),"simple_mean_loss":float(df.test_loss.mean()),"weighted_auc":float(np.average(df.test_auc,weights=df.n_samples)),"weighted_acc":float(np.average(df.test_acc,weights=df.n_samples))}

def run_datasets_v2(root,selected=None,debug=False,num_epochs=NUM_EPOCHS,resume=True):
    base=model_output_dir(debug); paths=ensure_output_tree(base); final=base/"final_results.csv"; histories=base/"histories.csv"; failed=base/"failed_datasets.csv"
    atomic_write_json(run_config(debug,num_epochs),base/"run_config.json"); available=discover_datasets(root); names=available if selected is None else [n for n in selected if n in available]
    done=set()
    if resume and final.exists():
        old=pd.read_csv(final); done=set(old.loc[old.status=="completed","dataset"]) if "status" in old else set()
    print(f"Output directory: {base}"); print(f"Datasets requested: {len(names)}"); print(f"Already completed and skipped: {len(done.intersection(names))}")
    for i,name in enumerate(names,1):
        if name in done: print(f"[{i}/{len(names)}] Skip completed: {name}"); continue
        print(f"\n[{i}/{len(names)}] Start: {name}")
        try:
            result,hist=run_one_dataset_v2(root/name,name,paths,num_epochs,SAVE_INDIVIDUAL_CURVES); append_or_replace(final,pd.DataFrame([result]),name); append_or_replace(histories,hist,name)
            if failed.exists(): atomic_write_csv(pd.read_csv(failed).query("dataset != @name"),failed)
        except Exception as e:
            failure=pd.DataFrame([{"dataset":name,"model":MODEL_NAME,"seed":SEED,"error_type":type(e).__name__,"error_message":str(e),"traceback":traceback.format_exc(),"recorded_utc":datetime.now(timezone.utc).isoformat()}]); append_or_replace(failed,failure,name); print(f"FAILED {name}: {type(e).__name__}: {e}")
    if not final.exists(): raise RuntimeError("No dataset completed successfully.")
    out=pd.read_csv(final); out=out[out.dataset.isin(names)].sort_values("dataset").reset_index(drop=True); summary=summarize(out); summary.update({"model":MODEL_NAME,"seed":SEED,"debug":debug,"datasets_requested":len(names),"generated_utc":datetime.now(timezone.utc).isoformat()}); atomic_write_json(summary,base/"summary.json")
    print("\n========== FEDformer V2 SUMMARY (OFFICIAL TEST, EVALUATED ONCE) =========="); print(out[["dataset","test_auc","test_acc","test_loss","n_samples"]].to_string(index=False)); print(f"\nSimple mean: AUC={summary['simple_mean_auc']:.4f}, ACC={summary['simple_mean_acc']:.4f}, LOSS={summary['simple_mean_loss']:.4f}"); print(f"Weighted: AUC={summary['weighted_auc']:.4f}, ACC={summary['weighted_acc']:.4f}"); return out,summary

## Run order

1. Keep `RUN_SMOKE_TEST=True` and `RUN_FULL_EXPERIMENT=False`, then restart the kernel and run all cells. Coffee and ArrowHead should complete.
2. Confirm the audit says `uses_aout=False`, `decomposition=moving_average`, and `frequency_block=FourierEnhanced`.
3. Set `RUN_SMOKE_TEST=False`, `RUN_FULL_EXPERIMENT=True`, restart the kernel, and run all cells for the final experiment.

Smoke output: `final_runs_v2/_debug/fedformer/seed_42/`. Final output: `final_runs_v2/fedformer/seed_42/`.

In [ ]:
# Cell 7 - smoke and final switches
if RUN_SMOKE_TEST:
    smoke_results, smoke_summary = run_datasets_v2(DATA_ROOT, selected=SMOKE_DATASETS, debug=True, num_epochs=SMOKE_EPOCHS, resume=False)
else:
    print("Smoke test disabled.")
if RUN_FULL_EXPERIMENT:
    full_results, full_summary = run_datasets_v2(DATA_ROOT, selected=None, debug=False, num_epochs=NUM_EPOCHS, resume=RESUME_COMPLETED_DATASETS)
else:
    print("Full experiment disabled.")